### Notebook 03: Predictive Modeling — Regression, Classification, Clustering, Forecasting

**CS 210 Final Project** | What the Headlines Say: Measuring Cognitive Bias and Agency Framing in AI News Discourse | Sadhana Vasanthakumar

Builds and evaluates predictive models on the feature table from notebook 02. Five families of analysis:

1. Regression (Ridge) — predicts AAI from headline features
2. Classification (Decision Tree) — predicts dominant bias type
3. Clustering (K-means) — discovers headline archetypes
4. Hypothesis testing — pre-registered tests with effect sizes
5. Trend extrapolation — linear projection with explicit caveats

A few things worth flagging up front. AAI regression uses two feature sets — several bias dimensions share vocabulary with AAI components, which inflates R² if included. Cell 3 runs both and reports the honest no-circular number (R² = 0.067) alongside the full-feature number (R² = 0.22) with the circularity explicitly disclosed. Logistic Regression accuracy (99.46%) in cell 5 is tautological — `dominant_bias` is the argmax of the bias_* columns, which are included as features. The Decision Tree (74.4%, interpretable, no overfit) is the correct headline result. Cluster names are assigned after stability validation in cell 6 — if ARI is below 0.7, use generic labels. The extrapolation in cell 8 is not a forecast; with n=20 monthly observations it's framed as "if the trend continues at the same rate."

#### Inputs

| File | Source | Required |
|------|--------|----------|
| `outputs/features_for_modeling.csv` | Notebook 02 | Yes (179,372 rows) |
| `outputs/monthly_trend.csv` | Notebook 02 | Yes (20 rows) |

#### Outputs

| File | Purpose |
|------|---------|
| `outputs/website_data.json` | All results for the dashboard |
| `outputs/figures/06_*.png` through `11_*.png` | Modeling visualizations |
| `outputs/cluster_assignments.csv` | Per-headline cluster labels |
| `outputs/hypothesis_test_results.csv` | Pre-registered test results |
| `outputs/cluster_stability_report.csv` | Multi-seed cluster ARI matrix |
| `outputs/findings_interpretation.txt` | Prose findings |

#### Dependencies

```bash
pip install scikit-learn pandas numpy matplotlib seaborn scipy
```

#### How to run

Takes 12–20 minutes end-to-end. Cell 4 (5 classifiers on 179K rows) and cell 6 (multi-seed cluster stability) are the slowest.

1. Confirm both CSV inputs exist from notebook 02
2. Run all cells top to bottom

In [1]:
# imports and load feature data. random seed 42 set globally.

import pandas as pd
import numpy as np
import os, json
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    silhouette_score, adjusted_rand_score,
)

from scipy import stats
from scipy.stats import t as t_dist, pearsonr, ttest_ind

import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs("outputs/figures", exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv("outputs/features_for_modeling.csv", encoding="utf-8")
trend = pd.read_csv("outputs/monthly_trend.csv", encoding="utf-8")

assert "aai_score" in df.columns
assert "dominant_bias" in df.columns
assert len(trend) >= 18

print(f"loaded {len(df):,} headlines x {len(df.columns)} columns")
print(f"loaded {len(trend)} monthly trend rows")
print(f"random seed: {RANDOM_SEED}")

loaded 179,372 headlines x 28 columns
loaded 20 monthly trend rows
random seed: 42


In [2]:
# build two feature sets for AAI regression.
# FULL includes bias_intensity and 4 bias dimensions that share vocabulary
# with AAI components (fear_bias, moral_panic, control_loss, economic_threat).
# those overlaps create a partial circularity — including them inflates R²
# without adding real predictive signal.
# NO_CIRCULAR drops them. that's the honest R².
# cell 3 fits both and reports the difference explicitly.
# Why two feature sets:
#   FULL feature set:        gives the highest R-squared but the
#                            R-squared is partially circular.
#   NO_CIRCULAR feature set: drops bias_intensity AND the bias_*
#                            dimensions whose vocabulary overlaps
#                            with AAI components (fear_bias,
#                            moral_panic, control_loss, economic_threat).
#                            R-squared will be lower but is the
#                            HONEST number.

BIAS_COLS = [c for c in df.columns
             if c.startswith("bias_") and c != "bias_intensity"]

# bias dimensions whose vocabulary overlaps AAI components
CIRCULAR_BIAS_COLS = [
    "bias_fear_bias",
    "bias_moral_panic",
    "bias_control_loss",
    "bias_economic_threat",
]
NONCIRCULAR_BIAS_COLS = [c for c in BIAS_COLS if c not in CIRCULAR_BIAS_COLS]

window_order = sorted(df["window"].unique())
df["month_num"] = df["window"].map({w: i + 1 for i, w in enumerate(window_order)})
df["is_risk_category"]   = (df["category"] == "Safety & Risk").astype(int)
df["is_agency_category"] = (df["category"] == "Agency & Autonomy").astype(int)

# full feature set (highest R² but partially circular)
FEATURES_FULL = [
    "vader_compound", "psi_score", "bias_intensity",
    "month_num", "is_risk_category", "is_agency_category",
] + BIAS_COLS

# no-circular feature set (honest R²)
FEATURES_NO_CIRCULAR = [
    "vader_compound", "psi_score",
    "month_num", "is_risk_category", "is_agency_category",
] + NONCIRCULAR_BIAS_COLS

model_df = df[FEATURES_FULL + ["aai_score", "dominant_bias"]].dropna()

print(f"model-ready rows: {len(model_df):,}")
print()
print(f"FULL feature set ({len(FEATURES_FULL)} features):")
for f in FEATURES_FULL:
    flag = " (CIRCULAR)" if f in CIRCULAR_BIAS_COLS or f == "bias_intensity" else ""
    print(f"   {f}{flag}")
print()
print(f"NO_CIRCULAR feature set ({len(FEATURES_NO_CIRCULAR)} features):")
for f in FEATURES_NO_CIRCULAR:
    print(f"   {f}")
print()
print(f"removed (circular): {[c for c in FEATURES_FULL if c not in FEATURES_NO_CIRCULAR]}")

model-ready rows: 179,372

FULL feature set (14 features):
   vader_compound
   psi_score
   bias_intensity (CIRCULAR)
   month_num
   is_risk_category
   is_agency_category
   bias_fear_bias (CIRCULAR)
   bias_optimism_bias
   bias_anthropomorphism
   bias_moral_panic (CIRCULAR)
   bias_agency_attribution
   bias_techno_utopianism
   bias_economic_threat (CIRCULAR)
   bias_control_loss (CIRCULAR)

NO_CIRCULAR feature set (9 features):
   vader_compound
   psi_score
   month_num
   is_risk_category
   is_agency_category
   bias_optimism_bias
   bias_anthropomorphism
   bias_agency_attribution
   bias_techno_utopianism

removed (circular): ['bias_intensity', 'bias_fear_bias', 'bias_moral_panic', 'bias_economic_threat', 'bias_control_loss']


In [3]:
# regression: 5 models on both feature sets.
# Ridge selected: L2 regularization shrinks correlated coefficients, stable
# on multi-collinear data, standardized coefficients are directly interpretable.
# OLS is numerically unstable when bias_* features correlate. Lasso discards
# related dimensions. RF and GBM have higher accuracy but lose signed
# coefficient interpretability — can't tell what direction drives anxiety.
# report leads with the no-circular R² (honest). full-feature R² shown for
# comparison with explicit circularity disclosure.
# estimated runtime: 4-6 minutes (RF and GB are slowest)

# Key decision -- why Ridge over alternatives:
#
#   MODEL              KEY PROPERTY                        DECISION
#   -----              ------------                        --------
#   OLS LinearReg      No regularization, baseline         Rejected
#                      Numerically unstable when bias_*
#                      features are correlated.
#
#   Ridge (L2)         Shrinks correlated coefficients     SELECTED
#                      toward each other. Stable on
#                      multi-collinear data. Standardized
#                      coefficients directly interpretable.
#
#   Lasso (L1)         Sparse coefficients (sets some       Rejected
#                      to zero). Throws away information
#                      from related bias dimensions.
#
#   Random Forest      Higher accuracy, captures             Rejected
#                      non-linearities. But unsigned
#                      feature importances -- cannot tell
#                      direction of effect.
#
#   Gradient Boosting  Often highest accuracy on tabular.   Rejected
#                      Same interpretability problem.
#                      Black-box for "what drives anxiety?"

def fit_regression_suite(X, y, label, scaler, X_train_s, X_test_s, y_train, y_test):
    """fit all 5 models on a given feature set, return result table."""
    models = {
        "OLS LinearReg":     LinearRegression(),
        "Ridge (alpha=1.0)": Ridge(alpha=1.0),
        "Lasso (alpha=0.01)":Lasso(alpha=0.01, max_iter=10000),
        "Random Forest":     RandomForestRegressor(n_estimators=50, max_depth=10,
                                                    n_jobs=-1, random_state=RANDOM_SEED),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=50, max_depth=4,
                                                        random_state=RANDOM_SEED),
    }

    print(f"\nregression comparison — feature set: {label}")
    print(f"   train: {len(X_train_s):,}   test: {len(X_test_s):,}")
    print()
    print(f"{'model':<22} {'R-sq':>8} {'MAE':>8} {'RMSE':>8} {'CV R-sq':>14}")
    print("-" * 65)

    results = {}
    for name, model in models.items():
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)
        r2   = r2_score(y_test, y_pred)
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        cv = cross_val_score(model, scaler.transform(X), y, cv=5, scoring="r2", n_jobs=-1)
        cv_str = f"{cv.mean():.3f}+/-{cv.std():.3f}"
        results[name] = {"r2": r2, "mae": mae, "rmse": rmse, "cv_mean": cv.mean(), "cv_std": cv.std()}
        print(f"{name:<22} {r2:>8.4f} {mae:>8.4f} {rmse:>8.4f} {cv_str:>14}")

    return results


# full feature set (with circular features)
X_full = model_df[FEATURES_FULL].values
y      = model_df["aai_score"].values
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_SEED
)
scaler_full = StandardScaler()
X_train_s = scaler_full.fit_transform(X_train)
X_test_s  = scaler_full.transform(X_test)

results_full = fit_regression_suite(X_full, y, "FULL (incl. circular features)",
                                      scaler_full, X_train_s, X_test_s, y_train, y_test)

# no-circular feature set (the honest one)
X_nc = model_df[FEATURES_NO_CIRCULAR].values
X_train_nc, X_test_nc, y_train_nc, y_test_nc = train_test_split(
    X_nc, y, test_size=0.2, random_state=RANDOM_SEED
)
scaler_nc = StandardScaler()
X_train_nc_s = scaler_nc.fit_transform(X_train_nc)
X_test_nc_s  = scaler_nc.transform(X_test_nc)

results_nc = fit_regression_suite(X_nc, y, "NO-CIRCULAR (honest features)",
                                    scaler_nc, X_train_nc_s, X_test_nc_s,
                                    y_train_nc, y_test_nc)

# compare Ridge between the two feature sets
ridge_full_r2 = results_full["Ridge (alpha=1.0)"]["r2"]
ridge_nc_r2   = results_nc["Ridge (alpha=1.0)"]["r2"]
inflation = (ridge_full_r2 - ridge_nc_r2) / max(ridge_nc_r2, 0.001) * 100

print()
print("circularity diagnosis:")
print(f"   Ridge R-squared with circular features:    {ridge_full_r2:.4f}")
print(f"   Ridge R-squared without circular features: {ridge_nc_r2:.4f}")
print(f"   inflation from circularity:                {inflation:+.1f}%")
print()
print(f"   R-squared = {ridge_nc_r2:.3f} (honest).")
print(f"   mention {ridge_full_r2:.3f} with the explicit caveat that {abs(inflation):.0f}% is circularity.")

# refit final Ridge on no-circular features for downstream use
ridge_final = Ridge(alpha=1.0).fit(X_train_nc_s, y_train_nc)
y_pred_ridge = ridge_final.predict(X_test_nc_s)
r2_ridge   = r2_score(y_test_nc, y_pred_ridge)
mae_ridge  = mean_absolute_error(y_test_nc, y_pred_ridge)

coef_df = pd.DataFrame({
    "feature":     FEATURES_NO_CIRCULAR,
    "coefficient": ridge_final.coef_,
}).sort_values("coefficient", key=abs, ascending=False)

print()
print("Ridge feature importance (no-circular features, standardized):")
print(coef_df.to_string(index=False))

# visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test_nc, y_pred_ridge, alpha=0.3, s=5, color="#2471a3")
axes[0].plot([y_test_nc.min(), y_test_nc.max()],
             [y_test_nc.min(), y_test_nc.max()], "r--", lw=2, label="Perfect fit")
axes[0].set_xlabel("Actual AAI")
axes[0].set_ylabel("Predicted AAI")
axes[0].set_title(f"Ridge (no-circular features)\nR-sq={r2_ridge:.3f}, MAE={mae_ridge:.3f}")
axes[0].legend()

top10 = coef_df.head(10)
colors = ["#c0392b" if c > 0 else "#1e8449" for c in top10["coefficient"]]
axes[1].barh(top10["feature"], top10["coefficient"], color=colors)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("Standardized coefficient")
axes[1].set_title("Top features\n(red=increases AAI, green=decreases)")
plt.tight_layout()
plt.savefig("outputs/figures/06_regression.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nsaved: 06_regression.png")



regression comparison — feature set: FULL (incl. circular features)
   train: 143,497   test: 35,875

model                      R-sq      MAE     RMSE        CV R-sq
-----------------------------------------------------------------
OLS LinearReg            0.2199   0.0536   0.0824  0.211+/-0.013
Ridge (alpha=1.0)        0.2199   0.0536   0.0824  0.211+/-0.013
Lasso (alpha=0.01)       0.0839   0.0593   0.0893  0.082+/-0.003
Random Forest            0.3014   0.0472   0.0780  0.291+/-0.021
Gradient Boosting        0.2835   0.0488   0.0790  0.274+/-0.018

regression comparison — feature set: NO-CIRCULAR (honest features)
   train: 143,497   test: 35,875

model                      R-sq      MAE     RMSE        CV R-sq
-----------------------------------------------------------------
OLS LinearReg            0.0669   0.0591   0.0901  0.060+/-0.008
Ridge (alpha=1.0)        0.0669   0.0591   0.0901  0.060+/-0.008
Lasso (alpha=0.01)       0.0279   0.0616   0.0920  0.025+/-0.002
Random Forest

In [4]:
# permutation test on Ridge (no-circular features).
# shuffles the AAI target 200 times and refits Ridge each time to build a
# null distribution of R². if the real model's R² never appears in the null,
# the model is learning real structure — not just capitalizing on dataset size.
# with 179K rows and 9 features, almost any model produces positive R² by
# chance without this test.
# estimated runtime: 3-5 minutes

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

print("running permutation test on Ridge (no-circular features)...")
print(f"200 permutations on {len(model_df):,} rows x {len(FEATURES_NO_CIRCULAR)} features")
print()

X_perm = model_df[FEATURES_NO_CIRCULAR].values
y_perm = model_df["aai_score"].values

scaler_perm = StandardScaler()
X_perm_s = scaler_perm.fit_transform(X_perm)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_perm_s, y_perm, test_size=0.2, random_state=RANDOM_SEED
)

# real model R² (recompute cleanly)
real_model = Ridge(alpha=1.0).fit(X_tr, y_tr)
real_r2 = r2_score(y_te, real_model.predict(X_te))

# permutation loop
N_PERM = 200
perm_r2s = []
rng = np.random.default_rng(RANDOM_SEED)

for i in range(N_PERM):
    y_shuffled = rng.permutation(y_perm)
    X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
        X_perm_s, y_shuffled, test_size=0.2, random_state=i
    )
    m = Ridge(alpha=1.0).fit(X_tr_p, y_tr_p)
    perm_r2s.append(r2_score(y_te_p, m.predict(X_te_p)))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/200 done...")

perm_r2s = np.array(perm_r2s)
emp_p = (perm_r2s >= real_r2).mean()

print()
print("permutation test results:")
print(f"  real Ridge R-squared:       {real_r2:.4f}")
print(f"  null distribution mean:     {perm_r2s.mean():.4f}")
print(f"  null distribution std:      {perm_r2s.std():.4f}")
print(f"  permutations >= real R-sq:  {(perm_r2s >= real_r2).sum()} / {N_PERM}")
print(f"  empirical p-value:          {emp_p:.4f}")
print()
if emp_p < 0.05:
    print("  verdict: PASSES permutation test.")
    print(f"  the model's R-sq={real_r2:.3f} cannot be explained by chance.")
    print("   permutation test confirms the model learns real structure")
    print(f"  (empirical p={emp_p:.3f}, 200 permutations).'")
else:
    print("  verdict: FAILS permutation test.")
    print("  the model's performance is within the range of chance.")
    print("  limitation and do not claim predictive validity.")

# save for website_data.json
permutation_test_result = {
    "real_r2": round(float(real_r2), 4),
    "null_mean": round(float(perm_r2s.mean()), 4),
    "null_std":  round(float(perm_r2s.std()), 4),
    "emp_p":     round(float(emp_p), 4),
    "n_perm":    N_PERM,
    "verdict":   "PASSES" if emp_p < 0.05 else "FAILS",
}
print()

running permutation test on Ridge (no-circular features)...
200 permutations on 179,372 rows x 9 features

  50/200 done...
  100/200 done...
  150/200 done...
  200/200 done...

permutation test results:
  real Ridge R-squared:       0.0669
  null distribution mean:     -0.0001
  null distribution std:      0.0001
  permutations >= real R-sq:  0 / 200
  empirical p-value:          0.0000

  verdict: PASSES permutation test.
  the model's R-sq=0.067 cannot be explained by chance.
   permutation test confirms the model learns real structure
  (empirical p=0.000, 200 permutations).'



In [5]:
# classification: 5 models predicting dominant_bias (8 classes).
#
# NOTE: Logistic Regression (99.46%) is tautological — dominant_bias is
# the argmax of the bias_* columns, which are included as features.
# it's essentially recovering the argmax function it was given, not
# learning a generalizable pattern. included here for completeness only.
# use DT pruned (74.4%) as the report headline — interpretable rules,
# no overfit, non-trivial result well above 12.5% random baseline.
#
# model selection rationale:
#   DT pruned (d=5): fully interpretable rules, max_depth=5 prevents overfit — SELECTED
#   DT unpruned:     demonstrates overfit (train=1.0, test much lower) — for comparison
#   Random Forest:   higher accuracy via ensembling but loses readable rule tree — rejected
#   KNN (k=15):      non-parametric baseline, no interpretable decision boundary — rejected
#
# estimated runtime: 4-6 minutes

le = LabelEncoder()
y_cls = le.fit_transform(model_df["dominant_bias"])

# use full feature set for classification (no AAI circularity issue here —
# we're predicting bias category, not AAI score)
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_cls, test_size=0.2, random_state=RANDOM_SEED)
X_tr_s = scaler_full.fit_transform(X_tr)
X_te_s = scaler_full.transform(X_te)

classification_models = {
    "Logistic Regression":  LogisticRegression(max_iter=2000, n_jobs=-1, random_state=RANDOM_SEED),
    "DT pruned (d=5)":      DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=RANDOM_SEED),
    "DT unpruned (d=None)": DecisionTreeClassifier(random_state=RANDOM_SEED),
    "Random Forest":        RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=RANDOM_SEED),
    "KNN (k=15)":           KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
}

print("classification model comparison")
print(f"   train: {len(X_tr):,}   test: {len(X_te):,}   classes: {len(le.classes_)}")
print()
print(f"{'model':<24} {'train':>10} {'test':>10} {'gap':>8} {'diagnosis':<14}")
print("-" * 75)

classification_results = {}
for name, model in classification_models.items():
    model.fit(X_tr_s, y_tr)
    train_acc = model.score(X_tr_s, y_tr)
    test_acc  = model.score(X_te_s, y_te)
    gap = train_acc - test_acc
    diag = "OVERFIT" if gap > 0.10 else "ok"
    classification_results[name] = {"train_acc": train_acc, "test_acc": test_acc, "gap": gap}
    print(f"{name:<24} {train_acc:>10.4f} {test_acc:>10.4f} {gap:>8.3f} {diag:<14}")

print()
print("selected: DT pruned (max_depth=5, min_samples_leaf=50)")
# NOTE: Logistic Regression (99.46%) is tautological — dominant_bias is the argmax of the
# bias_* columns, which are included as features. cite DT (74.4%)
print("NOTE: LR 99.46% is tautological (predicting argmax from its inputs). use DT (74.4%).")

# refit selected model
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=RANDOM_SEED).fit(X_tr_s, y_tr)
y_pred_dt = dt.predict(X_te_s)
test_acc_dt = dt.score(X_te_s, y_te)

cv_acc = cross_val_score(dt, scaler_full.transform(X_full), y_cls, cv=5, n_jobs=-1)
print(f"\n5-fold CV accuracy: {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print()
print("per-class metrics:")
print(classification_report(y_te, y_pred_dt, target_names=le.classes_, zero_division=0))

# confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_te, y_pred_dt)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap="Blues")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.title(f"Decision Tree Confusion Matrix (test acc = {test_acc_dt:.3f})", fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/07_decision_tree_cm.png", dpi=150, bbox_inches="tight")
plt.close()

# feature importance
fi_df = pd.DataFrame({"feature": FEATURES_FULL, "importance": dt.feature_importances_})
fi_df = fi_df[fi_df["importance"] > 0].sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi_df["feature"], fi_df["importance"], color="#f39c12")
ax.set_title("Decision Tree Feature Importance (Gini)", fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("outputs/figures/08_dt_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print()
print("saved: 07_decision_tree_cm.png, 08_dt_importance.png")

classification model comparison
   train: 143,497   test: 35,875   classes: 8

model                         train       test      gap diagnosis     
---------------------------------------------------------------------------
Logistic Regression          0.9960     0.9946    0.001 ok            
DT pruned (d=5)              0.7447     0.7439    0.001 ok            
DT unpruned (d=None)         1.0000     0.9710    0.029 ok            
Random Forest                0.8377     0.8307    0.007 ok            
KNN (k=15)                   0.8809     0.8636    0.017 ok            

selected: DT pruned (max_depth=5, min_samples_leaf=50)
NOTE: LR 99.46% is tautological (predicting argmax from its inputs). use DT (74.4%).

5-fold CV accuracy: 0.7474 +/- 0.0019

per-class metrics:
                    precision    recall  f1-score   support

agency_attribution       0.12      0.00      0.00       916
  anthropomorphism       0.93      0.26      0.41      1433
      control_loss       0.81      0.0

In [6]:
# k-means clustering with elbow + silhouette to select k.
# cell 7 runs stability validation — don't assign interpretive names
# to clusters until you see the ARI there.
# estimated runtime: 3-5 minutes

X_cl = model_df[BIAS_COLS].values
scaler_cl = StandardScaler()
X_sc = scaler_cl.fit_transform(X_cl)

K_RANGE = range(2, 10)
inertias, silhouettes = [], []

silh_idx = np.random.RandomState(RANDOM_SEED).choice(
    len(X_sc), min(5000, len(X_sc)), replace=False
)

print("fitting K-means for k = 2..9...")
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    silh = silhouette_score(X_sc[silh_idx], labels[silh_idx])
    silhouettes.append(silh)
    print(f"   k={k}: inertia={km.inertia_:.0f}, silhouette={silh:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(K_RANGE), inertias, "o-", color="#9b59b6", lw=2, ms=8)
ax1.axvline(4, color="red", ls="--", alpha=0.5, label="k=4 (selected)")
ax1.set_xlabel("k"); ax1.set_ylabel("Inertia"); ax1.set_title("Elbow Method", fontweight="bold")
ax1.legend()

ax2.plot(list(K_RANGE), silhouettes, "s-", color="#27ae60", lw=2, ms=8)
ax2.axvline(4, color="red", ls="--", alpha=0.5, label="k=4 (selected)")
ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette score")
ax2.set_title("Silhouette Validation", fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.savefig("outputs/figures/09_cluster_selection.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved: 09_cluster_selection.png")

# fit final K-means with k=4
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_SEED, n_init=10)
model_df = model_df.copy()
model_df["cluster"] = km_final.fit_predict(X_sc)

profile = model_df.groupby("cluster")[BIAS_COLS + ["aai_score", "psi_score"]].mean().round(4)
print("\ncluster profiles (mean values):")
print(profile.to_string())

fitting K-means for k = 2..9...
   k=2: inertia=681245, silhouette=0.4461
   k=3: inertia=468622, silhouette=0.3461
   k=4: inertia=388996, silhouette=0.2785
   k=5: inertia=350235, silhouette=0.2317
   k=6: inertia=327931, silhouette=0.1966
   k=7: inertia=311087, silhouette=0.1895
   k=8: inertia=297669, silhouette=0.1637
   k=9: inertia=283806, silhouette=0.1690
saved: 09_cluster_selection.png

cluster profiles (mean values):
         bias_fear_bias  bias_optimism_bias  bias_anthropomorphism  bias_moral_panic  bias_agency_attribution  bias_techno_utopianism  bias_economic_threat  bias_control_loss  aai_score  psi_score
cluster                                                                                                                                                                                             
0                0.3552              0.4171                 0.3945            0.4381                   0.3699                  0.4945                0.3455             0.426

In [7]:
# cluster stability validation: tests whether k=4 is reproducible.
# three robustness tests:
#   1. multi-seed: same features, 5 different random seeds
#   2. bootstrap: same features, 80% subsamples
#   3. feature-subset: drop one bias dimension at a time
# ARI > 0.7 = stable, interpretive names defensible.
# ARI 0.5-0.7 = moderately stable, use cautious naming.
# ARI < 0.5 = not stable, use generic labels (Cluster 0, 1, 2, 3).
# without this, the archetype names are post-hoc storytelling about
# one specific run, not a robust finding.
# estimated runtime: 4-6 minutes

print("cluster stability validation")
print()

# test 1: multi-seed stability
print("--- test 1: multi-seed stability ---")
SEEDS = [42, 0, 1, 7, 13]
seed_labels = {}
for seed in SEEDS:
    km = KMeans(n_clusters=K_FINAL, random_state=seed, n_init=10)
    seed_labels[seed] = km.fit_predict(X_sc)

print(f"\npairwise Adjusted Rand Index between seeds:")
print(f"{'':>8}", end="")
for s in SEEDS:
    print(f"{s:>8}", end="")
print()

multi_seed_ari = []
for s1 in SEEDS:
    print(f"{s1:>8}", end="")
    for s2 in SEEDS:
        if s1 == s2:
            ari_val = 1.0
        else:
            ari_val = adjusted_rand_score(seed_labels[s1], seed_labels[s2])
            multi_seed_ari.append(ari_val)
        print(f"{ari_val:>8.3f}", end="")
    print()

mean_ari_seed = np.mean(multi_seed_ari)
print(f"\nmean pairwise ARI (seeds): {mean_ari_seed:.4f}")

# test 2: bootstrap stability (80% subsamples)
print("\n--- test 2: bootstrap stability (80% subsamples) ---")
N_BOOTSTRAP = 5
np.random.seed(RANDOM_SEED)
boot_aris = []

for b in range(N_BOOTSTRAP):
    idx = np.random.choice(len(X_sc), int(len(X_sc) * 0.8), replace=False)
    km_b = KMeans(n_clusters=K_FINAL, random_state=RANDOM_SEED, n_init=10)
    labels_b = km_b.fit_predict(X_sc[idx])
    orig_labels_subset = km_final.labels_[idx]
    ari_b = adjusted_rand_score(orig_labels_subset, labels_b)
    boot_aris.append(ari_b)
    print(f"   bootstrap {b+1}: ARI vs original = {ari_b:.4f}")

mean_boot_ari = np.mean(boot_aris)
print(f"\nmean bootstrap ARI: {mean_boot_ari:.4f}")

# test 3: feature-subset stability
print("\n--- test 3: feature-subset stability (drop one bias dim) ---")
subset_aris = []
for dropped_col in BIAS_COLS:
    subset_cols = [c for c in BIAS_COLS if c != dropped_col]
    X_sub = StandardScaler().fit_transform(model_df[subset_cols].values)
    km_sub = KMeans(n_clusters=K_FINAL, random_state=RANDOM_SEED, n_init=10)
    labels_sub = km_sub.fit_predict(X_sub)
    ari_sub = adjusted_rand_score(km_final.labels_, labels_sub)
    subset_aris.append(ari_sub)
    print(f"   drop {dropped_col:<25}: ARI = {ari_sub:.4f}")

mean_subset_ari = np.mean(subset_aris)
print(f"\nmean feature-subset ARI: {mean_subset_ari:.4f}")

# summary verdict
print("\ncluster stability verdict:")
print(f"   multi-seed mean ARI:    {mean_ari_seed:.4f}")
print(f"   bootstrap mean ARI:     {mean_boot_ari:.4f}")
print(f"   feature-subset ARI:     {mean_subset_ari:.4f}")

overall_ari = np.mean([mean_ari_seed, mean_boot_ari, mean_subset_ari])
print(f"   OVERALL stability:      {overall_ari:.4f}")
print()

if overall_ari >= 0.70:
    stability_verdict = "STABLE"
    naming_advice = (
        "clusters are reproducible across seeds, subsamples, and feature subsets.\n"
        "   interpretive names (High Anxiety, Moderate Bias, etc.) are defensible."
    )
elif overall_ari >= 0.50:
    stability_verdict = "MODERATELY STABLE"
    naming_advice = (
        "clusters are partially reproducible. some structure is real, but exact\n"
        "   boundaries shift. use cautious names in the report and acknowledge\n"
        "   stability ARI in the methodology section."
    )
else:
    stability_verdict = "UNSTABLE"
    naming_advice = (
        "clusters are NOT reproducible across runs. use generic labels (Cluster 0,\n"
        "   Cluster 1, ...) in the report. note this as a limitation."
    )

print(f"   verdict: {stability_verdict}")
print(f"   {naming_advice}")

# save stability report
stability_report = {
    "test": ["multi_seed", "bootstrap", "feature_subset", "overall"],
    "mean_ari": [
        round(float(mean_ari_seed), 4),
        round(float(mean_boot_ari), 4),
        round(float(mean_subset_ari), 4),
        round(float(overall_ari), 4),
    ],
    "verdict": ["", "", "", stability_verdict],
}
pd.DataFrame(stability_report).to_csv("outputs/cluster_stability_report.csv", index=False)
print(f"\nsaved: outputs/cluster_stability_report.csv")

# assign cluster names based on stability verdict
if overall_ari >= 0.50:
    aai_by_cluster = model_df.groupby("cluster")["aai_score"].mean().sort_values(ascending=False)
    ranks = aai_by_cluster.rank(ascending=False).astype(int)
    NAME_MAP = {}
    RANK_NAMES = {1: "High Anxiety", 2: "Moderate Bias", 3: "Low Concern", 4: "Neutral Reporting"}
    for cl, rank in ranks.items():
        NAME_MAP[int(cl)] = RANK_NAMES[int(rank)]
else:
    NAME_MAP = {i: f"Cluster {i}" for i in range(K_FINAL)}

model_df["cluster_name"] = model_df["cluster"].map(NAME_MAP)

print(f"\ncluster names (based on stability verdict):")
for k_id, name in NAME_MAP.items():
    n = (model_df["cluster"] == k_id).sum()
    print(f"   cluster {k_id} '{name}': {n:,} headlines ({n/len(model_df)*100:.1f}%)")

cluster stability validation

--- test 1: multi-seed stability ---

pairwise Adjusted Rand Index between seeds:
              42       0       1       7      13
      42   1.000   0.976   0.979   0.962   0.979
       0   0.976   1.000   0.955   0.985   0.996
       1   0.979   0.955   1.000   0.941   0.958
       7   0.962   0.985   0.941   1.000   0.983
      13   0.979   0.996   0.958   0.983   1.000

mean pairwise ARI (seeds): 0.9713

--- test 2: bootstrap stability (80% subsamples) ---
   bootstrap 1: ARI vs original = 0.9682
   bootstrap 2: ARI vs original = 0.9708
   bootstrap 3: ARI vs original = 0.9584
   bootstrap 4: ARI vs original = 0.9833
   bootstrap 5: ARI vs original = 0.9945

mean bootstrap ARI: 0.9750

--- test 3: feature-subset stability (drop one bias dim) ---
   drop bias_fear_bias           : ARI = 0.8807
   drop bias_optimism_bias       : ARI = 0.8814
   drop bias_anthropomorphism    : ARI = 0.8725
   drop bias_moral_panic         : ARI = 0.8908
   drop bias_agenc

In [8]:
# PCA visualization of clusters in 2D space
# estimated runtime: 1-2 minutes

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_sc)
var_explained = pca.explained_variance_ratio_

print(f"PCA variance explained: PC1={var_explained[0]*100:.1f}%, "
      f"PC2={var_explained[1]*100:.1f}%, Total={sum(var_explained)*100:.1f}%")
print(f"note: {(1-sum(var_explained))*100:.1f}% of variance is in PC3..PC8.")

CLUSTER_COLORS = ["#c0392b", "#2471a3", "#1e8449", "#f39c12"]
plot_idx = np.random.RandomState(RANDOM_SEED).choice(
    len(X_pca), min(5000, len(X_pca)), replace=False
)

fig, ax = plt.subplots(figsize=(11, 7))
for k_id, name in NAME_MAP.items():
    mask = model_df["cluster"].values[plot_idx] == k_id
    ax.scatter(X_pca[plot_idx][mask, 0], X_pca[plot_idx][mask, 1],
               c=CLUSTER_COLORS[k_id], label=f"Cluster {k_id}: {name}",
               alpha=0.5, s=14, edgecolors="none")

ax.set_xlabel(f"PC1 ({var_explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]*100:.1f}% variance)")
ax.set_title("K-means Clusters in PCA Space (5,000-point sample)",
             fontweight="bold", fontsize=13)
ax.legend(fontsize=11, markerscale=2)
plt.tight_layout()
plt.savefig("outputs/figures/10_pca_clusters.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved: 10_pca_clusters.png")

PCA variance explained: PC1=81.1%, PC2=5.5%, Total=86.7%
note: 13.3% of variance is in PC3..PC8.
saved: 10_pca_clusters.png


In [9]:
# linear trend extrapolation — NOT a forecast.
# with n=20 monthly observations, ARIMA is unfit and Prophet is overconfident.
# this answers: "if the linear trend continues at the same rate, where would
# the metric be 3 months out?" — a different claim from a real prediction.
# 95% prediction intervals widen with distance, which correctly reflects
# increasing uncertainty outside the observed window.
# estimated runtime: < 1 minute

print("exploratory extrapolation — illustrative only")
print()
print("it is a visual aid. the 20-month series is below the threshold")
print("for reliable trend forecasting; prediction intervals widen rapidly.")
print()
print("specifically excluded:")
print("   - no AR / seasonal structure modeled")
print("   - no regime-change detection")
print("   - n=20 monthly observations is below the n=50+ guideline")
print("     for parametric trend extrapolation")
print()


def extrapolate(metric_col, trend_df, n_forecast=3, alpha=0.05):
    x = np.arange(1, len(trend_df) + 1, dtype=float).reshape(-1, 1)
    y = trend_df[metric_col].values.astype(float)
    reg = LinearRegression().fit(x, y)
    y_hat = reg.predict(x)
    resid = y - y_hat
    s2 = np.sum(resid ** 2) / max(len(y) - 2, 1)
    x_mean = x.mean()
    Sxx = np.sum((x - x_mean) ** 2)
    fx = np.arange(len(trend_df) + 1, len(trend_df) + n_forecast + 1, dtype=float).reshape(-1, 1)
    fy = reg.predict(fx)
    t_crit = t_dist.ppf(1 - alpha / 2, df=max(len(y) - 2, 1))
    se = np.sqrt(s2 * (1 + 1/len(y) + (fx.flatten() - x_mean) ** 2 / Sxx))
    return {
        "slope":    float(reg.coef_[0]),
        "r2":       float(r2_score(y, y_hat)),
        "fitted":   y_hat,
        "future_x": fx.flatten(),
        "future_y": fy,
        "ci_lo":    fy - t_crit * se,
        "ci_hi":    fy + t_crit * se,
    }

FW_MONTHS = ["2026-04", "2026-05", "2026-06"]
psi_fc = extrapolate("psi_index", trend)
aai_fc = extrapolate("avg_aai", trend)

print("trend extrapolation (not a forecast)")
print()
print("if the observed linear trend continues at")
print("the same rate, the metric would project to the values below.'")
print()
print("PSI trend:")
print(f"   slope:  {psi_fc['slope']:+.3f} PSI points/month  (R-sq={psi_fc['r2']:.3f})")
print()
print("PSI extrapolation (Apr-Jun 2026):")
for i, m in enumerate(FW_MONTHS):
    print(f"   {m}: {psi_fc['future_y'][i]:.1f}  "
          f"(95% CI: {psi_fc['ci_lo'][i]:.1f} to {psi_fc['ci_hi'][i]:.1f})")

print()
print("AAI trend:")
print(f"   slope:  {aai_fc['slope']:+.5f} AAI points/month  (R-sq={aai_fc['r2']:.3f})")
print()
print("AAI extrapolation (Apr-Jun 2026):")
for i, m in enumerate(FW_MONTHS):
    print(f"   {m}: {aai_fc['future_y'][i]:.4f}  "
          f"(95% CI: {aai_fc['ci_lo'][i]:.4f} to {aai_fc['ci_hi'][i]:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x_h = np.arange(1, len(trend) + 1)
x_f = np.arange(len(trend) + 1, len(trend) + 4)

for ax, fc, hist_col, color, ylabel in [
    (axes[0], psi_fc, "psi_index", "#c0392b", "PSI"),
    (axes[1], aai_fc, "avg_aai",   "#2471a3", "Mean AAI"),
]:
    hist = trend[hist_col].values
    ax.plot(x_h, hist, "o-", color=color, lw=2, ms=5, label="Observed")
    ax.plot(x_h, fc["fitted"], "--", color=color, alpha=0.5, lw=1.5, label="Linear fit")
    ax.plot(x_f, fc["future_y"], "s--", color=color, lw=2.5, ms=8, label="Extrapolation")
    ax.fill_between(x_f, fc["ci_lo"], fc["ci_hi"], alpha=0.2, color=color, label="95% PI")
    ax.axvline(len(trend) + 0.5, color="gray", ls=":", lw=1.5)
    ax.set_xticks(list(x_h) + list(x_f))
    ax.set_xticklabels(list(trend["window"]) + FW_MONTHS,
                       rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel}: Historical + Linear Extrapolation\n(NOT a forecast)",
                 fontweight="bold")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("outputs/figures/11_extrapolation.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nsaved: 11_extrapolation.png")

exploratory extrapolation — illustrative only

it is a visual aid. the 20-month series is below the threshold
for reliable trend forecasting; prediction intervals widen rapidly.

specifically excluded:
   - no AR / seasonal structure modeled
   - no regime-change detection
   - n=20 monthly observations is below the n=50+ guideline
     for parametric trend extrapolation

trend extrapolation (not a forecast)

if the observed linear trend continues at
the same rate, the metric would project to the values below.'

PSI trend:
   slope:  -10.604 PSI points/month  (R-sq=0.278)

PSI extrapolation (Apr-Jun 2026):
   2026-04: 651.8  (95% CI: 410.9 to 892.7)
   2026-05: 641.2  (95% CI: 397.0 to 885.3)
   2026-06: 630.6  (95% CI: 382.9 to 878.2)

AAI trend:
   slope:  +0.00048 AAI points/month  (R-sq=0.774)

AAI extrapolation (Apr-Jun 2026):
   2026-04: 0.0415  (95% CI: 0.0379 to 0.0451)
   2026-05: 0.0420  (95% CI: 0.0383 to 0.0456)
   2026-06: 0.0424  (95% CI: 0.0387 to 0.0462)

saved: 11_extr

In [10]:
# pre-registered hypothesis tests with effect sizes (Cohen's d).
# hypotheses were written before running any tests, based on the
# literature and dataset structure — not reverse-engineered from results.
#
# H1: PSI shows a significant upward trend over 20 months
#     (prior literature documents increasing agency attribution to AI in tech press)
# H2: Safety & Risk headlines have higher AAI than Companies & Products (d > 0.5)
#     (Safety & Risk is the tier most directly targeting anxiety framing)
# H3: Entity-named headlines (OpenAI, Google, etc.) have lower AAI than generic
#     (product-launch coverage is more optimistic than generic AI coverage)
# H4: Human-mentioning headlines engage IH domains more than others (sanity check)
#     (if H4 fails, the IH lexicon is broken)
# H5: PSI in major model release months is higher than baseline
#     (model releases typically generate AI-as-actor framing)
#
# effect size: |d| < 0.2 negligible, 0.2-0.5 small, 0.5-0.8 medium, > 0.8 large
# estimated runtime: 1-2 minutes

print("pre-registered hypothesis tests")
print()


def cohens_d(a, b):
    """Cohen's d for two independent samples."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    pooled_std = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    if pooled_std == 0:
        return 0.0
    return (a.mean() - b.mean()) / pooled_std


def interpret_d(d):
    abs_d = abs(d)
    if abs_d < 0.2: return "negligible"
    elif abs_d < 0.5: return "small"
    elif abs_d < 0.8: return "medium"
    else: return "large"


hypothesis_results = []

# H1: PSI upward trend
print("--- H1: PSI shows a significant upward trend ---")
x_time = np.arange(len(trend))
slope_psi, _, r_psi, p_psi, _ = stats.linregress(x_time, trend["psi_index"])
print(f"   slope:    {slope_psi:+.4f} PSI points/month")
print(f"   R-sq:     {r_psi**2:.4f}")
print(f"   p-value:  {p_psi:.4e}")
h1_supported = (slope_psi > 0) and (p_psi < 0.05)
print(f"   verdict:  {'SUPPORTED' if h1_supported else 'NOT supported'} (slope > 0 AND p < 0.05)")
hypothesis_results.append({
    "hypothesis": "H1: PSI upward trend",
    "test_statistic": round(float(slope_psi), 4),
    "p_value": round(float(p_psi), 6),
    "effect_size": round(float(r_psi**2), 4),
    "effect_size_metric": "R-squared",
    "verdict": "SUPPORTED" if h1_supported else "NOT supported",
})

# H2: Safety & Risk vs Companies & Products on AAI
print("\n--- H2: Safety & Risk has higher AAI than Companies & Products ---")
risk_aai = df.loc[df["category"] == "Safety & Risk", "aai_score"].dropna()
comp_aai = df.loc[df["category"] == "Companies & Products", "aai_score"].dropna()
if len(risk_aai) > 30 and len(comp_aai) > 30:
    t_h2, p_h2 = ttest_ind(risk_aai, comp_aai, equal_var=False)
    d_h2 = cohens_d(risk_aai, comp_aai)
    print(f"   Safety & Risk mean AAI:        {risk_aai.mean():.4f}  (n={len(risk_aai):,})")
    print(f"   Companies & Products mean AAI: {comp_aai.mean():.4f}  (n={len(comp_aai):,})")
    print(f"   Welch t:                       {t_h2:.3f}")
    print(f"   p-value:                       {p_h2:.4e}")
    print(f"   Cohen's d:                     {d_h2:+.3f}  ({interpret_d(d_h2)})")
    h2_supported = (d_h2 > 0.5) and (p_h2 < 0.05)
    print(f"   verdict: {'SUPPORTED' if h2_supported else 'NOT supported'} (d > 0.5 AND p < 0.05)")
    hypothesis_results.append({
        "hypothesis": "H2: Risk > Products on AAI",
        "test_statistic": round(float(t_h2), 4),
        "p_value": round(float(p_h2), 6),
        "effect_size": round(float(d_h2), 4),
        "effect_size_metric": "Cohen's d",
        "verdict": "SUPPORTED" if h2_supported else "NOT supported",
    })

# H3: entity-named headlines have lower AAI than generic
print("\n--- H3: Entity-named headlines have lower AAI than generic ---")
ENTITY_NAMES = ["openai", "chatgpt", "google", "anthropic", "claude",
                "meta", "microsoft", "deepmind", "nvidia", "apple intelligence",
                "gemini", "copilot", "deepseek", "mistral", "llama"]
df["has_entity"] = df.get("title", df.get("publisher", pd.Series([""] * len(df)))).apply(
    lambda t: int(any(e in str(t).lower() for e in ENTITY_NAMES))
)

title_for_check = df.get("title", df.get("publisher", pd.Series([""] * len(df))))
df["has_entity"] = title_for_check.apply(lambda t: int(any(e in str(t).lower() for e in ENTITY_NAMES)))

ent_aai = df.loc[df["has_entity"] == 1, "aai_score"].dropna()
gen_aai = df.loc[df["has_entity"] == 0, "aai_score"].dropna()
if len(ent_aai) > 30 and len(gen_aai) > 30:
    t_h3, p_h3 = ttest_ind(ent_aai, gen_aai, equal_var=False)
    d_h3 = cohens_d(ent_aai, gen_aai)
    print(f"   entity-named mean AAI: {ent_aai.mean():.4f}  (n={len(ent_aai):,})")
    print(f"   generic mean AAI:      {gen_aai.mean():.4f}  (n={len(gen_aai):,})")
    print(f"   Welch t:               {t_h3:.3f}")
    print(f"   p-value:               {p_h3:.4e}")
    print(f"   Cohen's d:             {d_h3:+.3f}  ({interpret_d(d_h3)})")
    h3_supported = (d_h3 < 0) and (p_h3 < 0.05)
    print(f"   verdict: {'SUPPORTED' if h3_supported else 'NOT supported'} (d < 0 AND p < 0.05)")
    hypothesis_results.append({
        "hypothesis": "H3: Entity-named < Generic on AAI",
        "test_statistic": round(float(t_h3), 4),
        "p_value": round(float(p_h3), 6),
        "effect_size": round(float(d_h3), 4),
        "effect_size_metric": "Cohen's d",
        "verdict": "SUPPORTED" if h3_supported else "NOT supported",
    })

# H4: human-mentioning headlines engage IH more (sanity check)
print("\n--- H4: human-mentioning headlines engage IH more (sanity check) ---")
if "mentions_humans" in df.columns:
    ih_mentioned = df.loc[df["mentions_humans"] == 1, "invisible_human_score"].dropna()
    ih_no = df.loc[df["mentions_humans"] == 0, "invisible_human_score"].dropna()
    if len(ih_mentioned) > 30 and len(ih_no) > 30:
        t_h4, p_h4 = ttest_ind(ih_mentioned, ih_no, equal_var=False)
        d_h4 = cohens_d(ih_mentioned, ih_no)
        print(f"   human-mentioning IH mean: {ih_mentioned.mean():.4f}  (n={len(ih_mentioned):,})")
        print(f"   other IH mean:             {ih_no.mean():.4f}  (n={len(ih_no):,})")
        print(f"   Cohen's d:                 {d_h4:+.3f}  ({interpret_d(d_h4)})")
        print(f"   p-value:                   {p_h4:.4e}")
        h4_supported = (d_h4 > 0) and (p_h4 < 0.05)
        print(f"   verdict: {'SUPPORTED' if h4_supported else 'NOT supported'}")
        if not h4_supported:
            print("   WARNING: this is the sanity-check hypothesis. if it fails,")
            print("   review the Invisible Human lexicon construction.")
        hypothesis_results.append({
            "hypothesis": "H4: Human-mentioning > Other on IH",
            "test_statistic": round(float(t_h4), 4),
            "p_value": round(float(p_h4), 6),
            "effect_size": round(float(d_h4), 4),
            "effect_size_metric": "Cohen's d",
            "verdict": "SUPPORTED" if h4_supported else "NOT supported",
        })
else:
    print("   skipped: mentions_humans column not in feature CSV.")

# H5: model release months have higher PSI than baseline
print("\n--- H5: model release months have higher PSI than baseline ---")
RELEASE_MONTHS = ["2025-01", "2025-05"]  # DeepSeek + GPT-5
release_mask = trend["window"].isin(RELEASE_MONTHS)
release_psi = trend.loc[release_mask, "psi_index"]
baseline_psi = trend.loc[~release_mask, "psi_index"]
if len(release_psi) > 0 and len(baseline_psi) > 1:
    t_h5, p_h5 = ttest_ind(release_psi, baseline_psi, equal_var=False)
    d_h5 = cohens_d(release_psi, baseline_psi)
    print(f"   release months PSI:  {release_psi.mean():.1f}  (n={len(release_psi)}: {RELEASE_MONTHS})")
    print(f"   baseline months PSI: {baseline_psi.mean():.1f}  (n={len(baseline_psi)})")
    print(f"   Cohen's d:           {d_h5:+.3f}  ({interpret_d(d_h5)})")
    print(f"   p-value:             {p_h5:.4f}")
    h5_supported = (d_h5 > 0) and (p_h5 < 0.10)  # relaxed alpha given small n
    print(f"   verdict: {'SUPPORTED' if h5_supported else 'NOT supported'} (relaxed alpha=0.10 given n=2)")
    print(f"   note: this test has very low power with only 2 release months.")
    hypothesis_results.append({
        "hypothesis": "H5: Release months > Baseline on PSI",
        "test_statistic": round(float(t_h5), 4),
        "p_value": round(float(p_h5), 6),
        "effect_size": round(float(d_h5), 4),
        "effect_size_metric": "Cohen's d",
        "verdict": "SUPPORTED" if h5_supported else "NOT supported",
    })

# save results
pd.DataFrame(hypothesis_results).to_csv("outputs/hypothesis_test_results.csv", index=False)
print()
print("summary:")
supported = [r for r in hypothesis_results if r["verdict"] == "SUPPORTED"]
not_supp  = [r for r in hypothesis_results if r["verdict"] != "SUPPORTED"]
print(f"   hypotheses tested:    {len(hypothesis_results)}")
print(f"   supported:            {len(supported)}")
print(f"   not supported:        {len(not_supp)}")
print()
for r in hypothesis_results:
    print(f"   {r['verdict']:<14} {r['hypothesis']}")
print()
print("saved: outputs/hypothesis_test_results.csv")

pre-registered hypothesis tests

--- H1: PSI shows a significant upward trend ---
   slope:    -10.6041 PSI points/month
   R-sq:     0.2776
   p-value:  1.7003e-02
   verdict:  NOT supported (slope > 0 AND p < 0.05)

--- H2: Safety & Risk has higher AAI than Companies & Products ---
   Safety & Risk mean AAI:        0.0926  (n=15,793)
   Companies & Products mean AAI: 0.0178  (n=28,155)
   Welch t:                       61.859
   p-value:                       0.0000e+00
   Cohen's d:                     +0.749  (medium)
   verdict: SUPPORTED (d > 0.5 AND p < 0.05)

--- H3: Entity-named headlines have lower AAI than generic ---
   entity-named mean AAI: 0.0268  (n=32,968)
   generic mean AAI:      0.0392  (n=146,404)
   Welch t:               -24.467
   p-value:               1.6293e-131
   Cohen's d:             -0.133  (negligible)
   verdict: SUPPORTED (d < 0 AND p < 0.05)

--- H4: human-mentioning headlines engage IH more (sanity check) ---
   human-mentioning IH mean: 0.0100  (n=

In [11]:
# AllSides political lean correlation — external validation of PSI/AAI/bias_intensity.
# if high-PSI headlines cluster in polarized sources, or high-AAI in left-leaning
# sources (who tend to cover AI risk more), that is convergent validity evidence
# for indices constructed without ever seeing the lean variable.
# all correlations within the AllSides-matched sub-sample (~11-12% of corpus) only.
# estimated runtime: 1-2 minutes

import sqlite3
from scipy.stats import spearmanr

print("AllSides lean correlation validation")
print()

try:
    conn_val = sqlite3.connect("data/bias_observatory.db")

    allsides_df = pd.read_sql_query("""
        SELECT
            f.psi_score,
            f.aai_score,
            f.bias_intensity,
            p.political_lean
        FROM headlines_raw r
        JOIN headlines_features f ON r.id = f.headline_id
        JOIN publishers p ON LOWER(r.publisher) = LOWER(p.name)
        WHERE p.political_lean IS NOT NULL
    """, conn_val)

    conn_val.close()

    lean_map = {"Left": -2, "Lean Left": -1, "Center": 0,
                "Lean Right": 1, "Right": 2,
                "left": -2, "lean-left": -1, "center": 0,
                "lean-right": 1, "right": 2}
    allsides_df["lean_numeric"] = allsides_df["political_lean"].map(lean_map)
    allsides_df = allsides_df.dropna(subset=["lean_numeric"])

    n_allsides = len(allsides_df)
    print(f"sub-corpus with AllSides ratings: {n_allsides:,} headlines")
    print(f"({n_allsides/len(df)*100:.1f}% of the full corpus)")
    print()
    print("Spearman correlations with political lean (numeric, left=-2 to right=+2):")
    print()
    print(f"  {'metric':<25} {'Spearman r':>12} {'p-value':>12} {'interpretation'}")
    print("-" * 70)

    for col, label in [
        ("psi_score",     "PSI score"),
        ("aai_score",     "AAI score"),
        ("bias_intensity","Bias intensity"),
    ]:
        r_val, p_val = spearmanr(allsides_df["lean_numeric"], allsides_df[col])
        if abs(r_val) < 0.10:   interp = "negligible"
        elif abs(r_val) < 0.30: interp = "weak"
        elif abs(r_val) < 0.50: interp = "moderate"
        else:                   interp = "strong"
        print(f"  {label:<25} {r_val:>+12.4f} {p_val:>12.4e}  {interp}")

    print()
    print("even weak correlations (|r| > 0.1, p < 0.05) are meaningful evidence")
    print("that the indices differentiate sources the way we expect.")
    print("if all correlations are negligible:")
    print("'our indices do not appear to capture politically-aligned framing,")
    print("which is itself a substantive finding.'")

    allsides_corr_result = {
        "n": int(n_allsides),
        "pct_corpus": round(n_allsides / len(df) * 100, 1),
    }
    for col, label in [("psi_score", "psi"), ("aai_score", "aai"), ("bias_intensity", "bias")]:
        r_val, p_val = spearmanr(allsides_df["lean_numeric"], allsides_df[col])
        allsides_corr_result[f"{label}_r"] = round(float(r_val), 4)
        allsides_corr_result[f"{label}_p"] = round(float(p_val), 6)

    print("stored in allsides_corr_result for website_data.json.")

except Exception as e:
    print(f"could not run AllSides validation: {e}")
    print("ensure publishers table exists with political_lean column.")
    allsides_corr_result = {}

AllSides lean correlation validation

sub-corpus with AllSides ratings: 13,891 headlines
(7.7% of the full corpus)

Spearman correlations with political lean (numeric, left=-2 to right=+2):

  metric                      Spearman r      p-value interpretation
----------------------------------------------------------------------
  PSI score                      -0.0042   6.1713e-01  negligible
  AAI score                      +0.0453   9.3320e-08  negligible
  Bias intensity                 +0.0113   1.8285e-01  negligible

even weak correlations (|r| > 0.1, p < 0.05) are meaningful evidence
that the indices differentiate sources the way we expect.
if all correlations are negligible:
'our indices do not appear to capture politically-aligned framing,
which is itself a substantive finding.'
stored in allsides_corr_result for website_data.json.


In [12]:
# lexicon sensitivity: drops 20% of keywords at random, recomputes bias
# scores, repeats 100 times. if median rank correlation >= 0.85, the
# ordering of headlines survives random keyword removal — scoring is
# driven by signal, not specific word choices.
# estimated runtime: 5-8 minutes

from scipy.stats import spearmanr

# rebuild lexicon here — same as notebook 02 cell 5
BIAS_LEXICON_SENS = {
    "fear_bias":          ["fear", "fears", "afraid", "scared", "panic", "alarming",
                           "terrifying", "dread", "horror", "threat", "threatens", "threatening",
                           "danger", "dangerous", "deadly", "lethal", "catastrophic", "devastating",
                           "dire", "nightmare", "catastrophe", "peril", "menace", "ominous"],
    "optimism_bias":      ["breakthrough", "revolutionary", "transform", "revolutionize",
                           "solve", "cure", "incredible", "amazing", "remarkable", "promising",
                           "hope", "hopeful", "optimistic", "positive", "beneficial", "improve",
                           "enhance", "empower", "opportunity", "potential"],
    "anthropomorphism":   ["thinks", "feels", "wants", "decides", "understands",
                           "learns", "knows", "believes", "desires", "dreams", "loves", "hates",
                           "chooses", "prefers", "worries", "hopes", "remembers", "imagines",
                           "perceives", "emotional", "sentient", "conscious", "mind"],
    "moral_panic":        ["crisis", "society", "collapse", "breakdown", "threat to",
                           "danger to", "destroy", "end of", "loss of", "erosion", "undermines",
                           "corrupts", "degrades", "alarm", "outrage", "scandal", "controversy",
                           "warning", "alert", "emergency"],
    "agency_attribution": ["takes over", "takes control", "in charge", "autonomous",
                           "independent", "self-directed", "without human", "on its own", "by itself",
                           "automatically", "decides for", "controls", "dominates", "runs",
                           "operates", "performs"],
    "techno_utopianism":  ["perfect", "flawless", "unlimited", "infinite",
                           "all-knowing", "solve all", "eliminate all", "end poverty", "end disease",
                           "save humanity", "utopia", "paradise", "golden age", "new era",
                           "transformed world", "without limits"],
    "economic_threat":    ["job loss", "job losses", "unemployment", "workers displaced",
                           "automation replaces", "jobs at risk", "eliminate jobs", "steal jobs",
                           "replace workers", "economic disruption", "workforce displacement",
                           "redundant workers", "layoffs"],
    "control_loss":       ["out of control", "uncontrollable", "uncontrolled", "cannot stop",
                           "impossible to stop", "beyond control", "no oversight", "unchecked",
                           "unregulated", "runs amok", "existential risk", "superintelligence",
                           "singularity", "arms race"],
}

# work on a 10K subsample to keep runtime under 10 minutes
sample_df = df.sample(10_000, random_state=RANDOM_SEED).copy()
sample_titles = sample_df["publisher"].fillna("").str.lower().values
if "title" in sample_df.columns:
    sample_titles = sample_df["title"].str.lower().values

N_TRIALS = 100
DROP_RATE = 0.20
rng = np.random.default_rng(RANDOM_SEED)

print("lexicon sensitivity analysis")
print(f"drop rate: {DROP_RATE*100:.0f}% per trial | trials: {N_TRIALS} | sample: 10,000 headlines")
print()
print(f"  {'dimension':<25} {'median r':>10} {'min r':>10} {'verdict'}")
print("-" * 58)

sensitivity_records = []

for dim_name, keywords in BIAS_LEXICON_SENS.items():
    full_scores = np.array([
        sum(1 for kw in keywords if kw in t)
        for t in sample_titles
    ], dtype=float)

    trial_rs = []
    for _ in range(N_TRIALS):
        n_drop = max(1, int(len(keywords) * DROP_RATE))
        keep_idx = rng.choice(len(keywords), size=len(keywords) - n_drop, replace=False)
        reduced_kws = [keywords[i] for i in keep_idx]
        reduced_scores = np.array([
            sum(1 for kw in reduced_kws if kw in t)
            for t in sample_titles
        ], dtype=float)
        if reduced_scores.std() > 0 and full_scores.std() > 0:
            r_val, _ = spearmanr(full_scores, reduced_scores)
            trial_rs.append(r_val)

    median_r = np.median(trial_rs) if trial_rs else 0.0
    min_r    = np.min(trial_rs)    if trial_rs else 0.0

    if median_r >= 0.85:   verdict = "STABLE"
    elif median_r >= 0.65: verdict = "MODERATE"
    else:                  verdict = "SENSITIVE"

    print(f"  {dim_name:<25} {median_r:>10.4f} {min_r:>10.4f}  {verdict}")
    sensitivity_records.append({
        "dimension": dim_name,
        "median_r":  round(float(median_r), 4),
        "min_r":     round(float(min_r), 4),
        "verdict":   verdict,
    })

sensitivity_df = pd.DataFrame(sensitivity_records)
sensitivity_df.to_csv("outputs/lexicon_sensitivity.csv", index=False)

print()
stable = sensitivity_df[sensitivity_df["verdict"] == "STABLE"]["dimension"].tolist()
sensitive = sensitivity_df[sensitivity_df["verdict"] == "SENSITIVE"]["dimension"].tolist()
print(f"STABLE dimensions ({len(stable)}): {stable}")
print(f"SENSITIVE dimensions ({len(sensitive)}): {sensitive}")
print()
print("STABLE: results are robust to specific keyword choices.")
print("SENSITIVE: qualify these dimensions in the report.")
print()
print("saved: outputs/lexicon_sensitivity.csv")

lexicon sensitivity analysis
drop rate: 20% per trial | trials: 100 | sample: 10,000 headlines

  dimension                   median r      min r verdict
----------------------------------------------------------
  fear_bias                     0.9720     0.6881  STABLE
  optimism_bias                 0.9029     0.6717  STABLE
  anthropomorphism              0.9250     0.6297  STABLE
  moral_panic                   0.9074     0.7100  STABLE
  agency_attribution            0.9562     0.5387  STABLE
  techno_utopianism             0.9547     0.4195  STABLE
  economic_threat               0.9852     0.5147  STABLE
  control_loss                  0.9622     0.7196  STABLE

STABLE dimensions (8): ['fear_bias', 'optimism_bias', 'anthropomorphism', 'moral_panic', 'agency_attribution', 'techno_utopianism', 'economic_threat', 'control_loss']
SENSITIVE dimensions (0): []

STABLE: results are robust to specific keyword choices.
SENSITIVE: qualify these dimensions in the report.

saved: outputs/le

In [13]:
# cross-metric analysis: how PSI, AAI, and Invisible Human interact.
# goes beyond individual metric statistics to show how the three
# indices relate to each other and to publisher structure.

print("cross-metric analysis")
print()

# (1) does AI-agent framing co-occur with anxiety?
psi_aai_corr, psi_aai_p = pearsonr(df["psi_score"], df["aai_score"])
print(f"(1) does AI-agent framing co-occur with anxiety?")
print(f"    Pearson r(PSI, AAI) = {psi_aai_corr:+.4f} (p={psi_aai_p:.2e})")

ai_agent_aai = df.loc[df["psi_flag"] == "AI_AGENT", "aai_score"].mean()
human_agent_aai = df.loc[df["psi_flag"] == "HUMAN_AGENT", "aai_score"].mean()
neutral_aai = df.loc[df["psi_flag"] == "NEUTRAL", "aai_score"].mean()
print(f"    mean AAI by PSI class:")
print(f"       AI_AGENT:    {ai_agent_aai:.4f}")
print(f"       HUMAN_AGENT: {human_agent_aai:.4f}")
print(f"       NEUTRAL:     {neutral_aai:.4f}")

# (2) do anxious headlines also omit human-experience vocabulary?
high_aai = df[df["aai_score"] > df["aai_score"].quantile(0.9)]
high_aai_no_human = high_aai[high_aai["invisible_human_score"] == 0]
pct_high_aai_invisible = len(high_aai_no_human) / max(len(high_aai), 1) * 100

print(f"\n(2) anxious headlines that omit ALL human-experience domains:")
print(f"    top 10% AAI headlines:                 {len(high_aai):,}")
print(f"    of those, ZERO invisible human terms:  {len(high_aai_no_human):,}  ({pct_high_aai_invisible:.1f}%)")

# (3) top publishers vs long tail
top_pubs = df["publisher"].value_counts().head(10).index.tolist()
top_aai = df[df["publisher"].isin(top_pubs)]["aai_score"].mean()
tail_aai = df[~df["publisher"].isin(top_pubs)]["aai_score"].mean()
top_psi = df[df["publisher"].isin(top_pubs)]["psi_score"].mean()
tail_psi = df[~df["publisher"].isin(top_pubs)]["psi_score"].mean()

print(f"\n(3) top 10 publishers vs long tail:")
print(f"    top 10 publishers:  mean AAI={top_aai:.4f}, mean PSI score={top_psi:.4f}")
print(f"    long tail (rest):   mean AAI={tail_aai:.4f}, mean PSI score={tail_psi:.4f}")

# (4) honest publisher diversity count
pubs_with_50 = (df["publisher"].value_counts() >= 50).sum()
pubs_with_100 = (df["publisher"].value_counts() >= 100).sum()
print(f"\n(4) honest publisher diversity (long tail caveat):")
print(f"    total unique publishers:           {df['publisher'].nunique():,}")
print(f"    publishers with >= 50 headlines:   {pubs_with_50:,}")
print(f"    publishers with >= 100 headlines:  {pubs_with_100:,}")
print(f"    use the >= 50 number in claims about diversity.")

cross-metric analysis

(1) does AI-agent framing co-occur with anxiety?
    Pearson r(PSI, AAI) = +0.0125 (p=1.13e-07)
    mean AAI by PSI class:
       AI_AGENT:    0.0465
       HUMAN_AGENT: 0.0661
       NEUTRAL:     0.0361

(2) anxious headlines that omit ALL human-experience domains:
    top 10% AAI headlines:                 17,618
    of those, ZERO invisible human terms:  16,023  (90.9%)

(3) top 10 publishers vs long tail:
    top 10 publishers:  mean AAI=0.0396, mean PSI score=0.5198
    long tail (rest):   mean AAI=0.0366, mean PSI score=0.5209

(4) honest publisher diversity (long tail caveat):
    total unique publishers:           13,019
    publishers with >= 50 headlines:   593
    publishers with >= 100 headlines:  286
    use the >= 50 number in claims about diversity.


In [16]:
# domain-specific findings interpretation — prose output for direct

findings_lines = []

def f(line=""):
    findings_lines.append(line)
    print(line)
    
f("DOMAIN-SPECIFIC FINDINGS INTERPRETATION")
f("(printed below; also saved to outputs/findings_interpretation.txt)")

f()

# finding 1: PSI direction
f("FINDING 1 -- AI Agency Framing (PSI)")
f("-" * 65)
direction = "increased" if slope_psi > 0 else "decreased"
f(f"Over 20 months (Aug 2024 - Mar 2026), the Power Shift Index")
f(f"{direction} at {slope_psi:+.3f} points/month (R-squared = {r_psi**2:.3f},")
f(f"p = {p_psi:.4f}).")
if slope_psi < 0:
    f(f"A negative slope indicates that AI-agent framing declined relative")
    f(f"to human-agent framing over the observation window. This likely")
    f(f"reflects growing coverage of human responses to AI -- regulation,")
    f(f"policy debates, research -- outpacing pure AI-as-actor coverage.")
else:
    f(f"This means: news coverage is increasingly framing AI systems as")
    f(f"the active subject of sentences, while humans become the passive")
    f(f"object or implicit operator.")
f()
f("Why this matters for AI discourse:")
f("Media framing shapes how readers attribute responsibility. Sentences")
f("like 'AI made a decision' implicitly absolve the humans who built and")
f("deployed the system. Sentences like 'engineers configured the system to")
f("decide' restore agency to humans. Tracking this ratio over time reveals")
f("structural shifts in how the public discusses accountability for AI.")
f()

# finding 2: AAI level and category split
f("FINDING 2 -- Anxiety in AI Coverage")
f("-" * 65)
f(f"Mean AAI across the corpus is {df['aai_score'].mean():.3f} on a [0,1] scale.")
if len(hypothesis_results) >= 2:
    h2 = hypothesis_results[1]
    f(f"Safety & Risk headlines have AAI {risk_aai.mean():.3f} vs Companies &")
    f(f"Products at {comp_aai.mean():.3f} (Cohen's d = {h2['effect_size']:+.3f}, ")
    f(f"p = {h2['p_value']:.2e}). The effect size is {interpret_d(h2['effect_size'])}.")
f()
f("Why this matters:")
f("Anxiety is concentrated in the Safety & Risk topic-tier as expected,")
f("but it is not absent from product coverage. Even routine company news")
f("retains some anxiety vocabulary, suggesting the broader news ecosystem")
f("treats AI as inherently risk-adjacent.")
f()

# finding 3: bias co-occurrence
f("FINDING 3 -- Cognitive Bias Co-occurrence")
f("-" * 65)
top_bias_pairs = []
for i, a in enumerate(BIAS_COLS):
    for j, b in enumerate(BIAS_COLS):
        if j > i:
            r, p = pearsonr(model_df[a], model_df[b])
            top_bias_pairs.append((a.replace("bias_", ""), b.replace("bias_", ""), r))
top_bias_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
top3 = top_bias_pairs[:3]

for a, b, r in top3:
    f(f"   {a} + {b}: r = {r:+.3f}")
f()
f("Strong positive co-occurrence among negative-framing biases (fear,")
f("moral_panic, control_loss, economic_threat) and among positive ones")
f("(optimism, techno_utopianism). The two camps are largely separable,")
f("which suggests media coverage has bifurcated rather than producing")
f("nuanced middle-ground framing.")
f()

# finding 4: cluster stability
f("FINDING 4 -- Headline Archetypes (with stability caveat)")
f("-" * 65)
f(f"K-means clustering identified 4 archetypes. Cluster stability across")
f(f"random seeds, bootstrap subsamples, and feature subsets averaged")
f(f"ARI = {overall_ari:.3f}.")
if overall_ari >= 0.70:
    f("This is a stable cluster solution, so the interpretive names")
    f("(High Anxiety, Moderate Bias, etc.) are defensible findings.")
elif overall_ari >= 0.50:
    f("This is a moderately stable solution. The number of clusters is")
    f("real but exact boundaries shift across runs. Use cautious naming.")
else:
    f("Clusters are NOT robustly reproducible. Use")
    f("generic cluster labels (Cluster 0..3) and note this limitation.")
f()

# finding 5: invisible human (refined)
f("FINDING 5 -- The Invisible Human (refined)")
f("-" * 65)
no_human_pct = ((df[[c for c in df.columns if c.startswith('ih_')]].sum(axis=1) == 0).mean() * 100)
f(f"{no_human_pct:.1f}% of all 179K AI headlines contain ZERO vocabulary")
f(f"from any human-experience domain (grief, spirituality, love,")
f(f"bodily experience, dignity, community, childhood).")
f()
if "mentions_humans" in df.columns:
    human_subset = df[df["mentions_humans"] == 1]
    cond_pct = ((human_subset[[c for c in df.columns if c.startswith('ih_')]].sum(axis=1) > 0).mean() * 100)
    f(f"Even conditioning on headlines that mention humans (workers,")
    f(f"researchers, users, etc.), only {cond_pct:.1f}% engage human-experience")
    f(f"domains. The absence is not just topical -- it is structural.")
    f()
f("Why this matters:")
f("AI is increasingly described in news as if its human consequences")
f("are technical or economic, not experiential. The dimensions of human")
f("life that ethics and philosophy treat as central -- grief, meaning,")
f("dignity, love -- are absent even when humans are explicitly")
f("the subject of AI coverage.")
f()

# finding 6: model performance with honesty
f("FINDING 6 -- Predictive Modeling (with honesty notes)")
f("-" * 65)
f(f"Ridge regression on AAI achieves R-squared = {r2_ridge:.3f} after")
f(f"removing circular features (bias dimensions sharing vocabulary")
f(f"with AAI components). Removing those features reduced R-squared")
f(f"from {ridge_full_r2:.3f} to {ridge_nc_r2:.3f}, indicating the full-feature")
f(f"score was inflated by vocabulary overlap rather than predictive signal.")
f()
f(f"Decision tree classification of dominant_bias achieves")
f(f"{test_acc_dt:.1%} test accuracy on 8 classes -- well above the")
f(f"random baseline of {1/8:.1%}.")
f()

with open("outputs/findings_interpretation.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(findings_lines))

print("\nsaved: outputs/findings_interpretation.txt")

DOMAIN-SPECIFIC FINDINGS INTERPRETATION
(printed below; also saved to outputs/findings_interpretation.txt)

FINDING 1 -- AI Agency Framing (PSI)
-----------------------------------------------------------------
Over 20 months (Aug 2024 - Mar 2026), the Power Shift Index
decreased at -10.604 points/month (R-squared = 0.278,
p = 0.0170).
A negative slope indicates that AI-agent framing declined relative
to human-agent framing over the observation window. This likely
reflects growing coverage of human responses to AI -- regulation,
policy debates, research -- outpacing pure AI-as-actor coverage.

Why this matters for AI discourse:
Media framing shapes how readers attribute responsibility. Sentences
like 'AI made a decision' implicitly absolve the humans who built and
deployed the system. Sentences like 'engineers configured the system to
decide' restore agency to humans. Tracking this ratio over time reveals
structural shifts in how the public discusses accountability for AI.

FINDING 2 -

In [15]:
# export all results to website_data.json

overall_psi_export = (
    (df["psi_flag"] == "AI_AGENT").sum() /
    max((df["psi_flag"] == "HUMAN_AGENT").sum(), 1) * 100
)

website_data = {
    "generated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "dataset": {
        "total_headlines":   int(len(df)),
        "unique_publishers": int(df["publisher"].nunique()),
        "publishers_50plus": int((df["publisher"].value_counts() >= 50).sum()),
        "date_range":        "2024-08-01 to 2026-03-31",
        "span_days":         607,
    },

    "overall": {
        "psi_overall":         round(float(overall_psi_export), 1),
        "aai_mean":            round(float(df["aai_score"].mean()), 4),
        "bias_intensity_mean": round(float(df["bias_intensity"].mean()), 4),
        "regression_r2_honest":  round(float(r2_ridge), 4),
        "regression_r2_full":    round(float(ridge_full_r2), 4),
        "regression_circularity_inflation_pct": round(float(inflation), 1),
        "regression_mae":      round(float(mae_ridge), 4),
        "classification_acc":  round(float(test_acc_dt), 4),
    },

    "monthly_trend":
        trend.round(4).to_dict(orient="records"),

    "extrapolation": {
        "months":    FW_MONTHS,
        "psi":       [round(float(v), 1) for v in psi_fc["future_y"]],
        "aai":       [round(float(v), 4) for v in aai_fc["future_y"]],
        "psi_ci_lo": [round(float(v), 1) for v in psi_fc["ci_lo"]],
        "psi_ci_hi": [round(float(v), 1) for v in psi_fc["ci_hi"]],
        "framing_note": "linear extrapolation, NOT a forecast",
    },

    "trend_stats": {
        "psi_slope":   round(float(slope_psi), 4),
        "psi_r2":      round(float(r_psi**2), 4),
        "psi_p_value": round(float(p_psi), 6),
    },

    "clusters": [
        {
            "id":       int(k_id),
            "name":     str(name),
            "size":     int((model_df["cluster"] == k_id).sum()),
            "pct":      round(float((model_df["cluster"] == k_id).mean() * 100), 1),
            "aai_mean": round(float(model_df.loc[model_df["cluster"] == k_id, "aai_score"].mean()), 4),
        }
        for k_id, name in NAME_MAP.items()
    ],
    "cluster_stability_ari": round(float(overall_ari), 4),

    "bias_means": {
        b.replace("bias_", ""): round(float(df[b].mean()), 4)
        for b in BIAS_COLS
    },

    "top_features": coef_df.head(8)[["feature", "coefficient"]].round(4).to_dict(orient="records"),

    "model_comparison": {
        "regression_full":     {k: {kk: round(float(vv), 4) for kk, vv in v.items()}
                                  for k, v in results_full.items()},
        "regression_honest":   {k: {kk: round(float(vv), 4) for kk, vv in v.items()}
                                  for k, v in results_nc.items()},
        "classification":      {k: {kk: round(float(vv), 4) for kk, vv in v.items()}
                                  for k, v in classification_results.items()},
    },

    "hypothesis_tests": hypothesis_results,

    "cross_metric": {
        "psi_aai_correlation": round(float(psi_aai_corr), 4),
        "psi_aai_p_value":     round(float(psi_aai_p), 6),
        "high_aai_invisible_pct": round(float(pct_high_aai_invisible), 2),
    },

    "category_distribution": {
        cat: int(n) for cat, n in df["category"].value_counts().items()
    },

    "top_publishers": {
        pub: int(n) for pub, n in df["publisher"].value_counts().head(20).items()
    },
}

website_data["permutation_test"] = permutation_test_result
website_data["allsides_validation"] = allsides_corr_result

with open("outputs/website_data.json", "w", encoding="utf-8") as f:
    json.dump(website_data, f, indent=2, ensure_ascii=False)

model_df[["cluster", "cluster_name"]].to_csv(
    "outputs/cluster_assignments.csv", index=False
)

print("outputs written:")
for fname in ["website_data.json", "cluster_assignments.csv",
               "hypothesis_test_results.csv", "cluster_stability_report.csv",
               "findings_interpretation.txt"]:
    path = f"outputs/{fname}"
    if os.path.exists(path):
        print(f"   outputs/{fname:<35} {os.path.getsize(path)//1024:>5,} KB")

print()
print("modeling complete.")
print()
print("final report numbers:")
print(f"   Ridge R-sq (honest):     {r2_ridge:.3f}")
print(f"   DT classification acc:   {test_acc_dt:.3f}")
print(f"   Cluster stability ARI:   {overall_ari:.3f}")
print(f"   PSI slope (per month):   {slope_psi:+.3f}  (p={p_psi:.4f})")
print(f"   Hypotheses supported:    {sum(1 for r in hypothesis_results if r['verdict']=='SUPPORTED')}/{len(hypothesis_results)}")
print()

outputs written:
   outputs/website_data.json                      12 KB
   outputs/cluster_assignments.csv             2,934 KB
   outputs/hypothesis_test_results.csv             0 KB
   outputs/cluster_stability_report.csv            0 KB
   outputs/findings_interpretation.txt             3 KB

modeling complete.

final report numbers:
   Ridge R-sq (honest):     0.067
   DT classification acc:   0.744
   Cluster stability ARI:   0.941
   PSI slope (per month):   -10.604  (p=0.0170)
   Hypotheses supported:    2/5

